[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nikbaya/smartbiomed_practicals_2026/blob/main/session1/practical.ipynb)

# Session 1: Introduction to GWAS — Practical

**Timing**: This practical is designed for ~45 minutes.
- Parts 1–3 should take ~30–35 minutes.
- Challenge questions are for fast finishers.

**Dataset**: Simulated GWAS data for ~100,000 variants with realistic block LD across chr1 (1–250 Mb).
- Continuous trait: simulated liability phenotype (h² ≈ 0.035, MAF-dependent spike-and-slab prior).
- Binary trait: derived from the continuous liability via a threshold model (~10% cases).

**Tip**: If you get stuck on any exercise, hints are in the comments. Solutions are in `answers.ipynb`.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy import stats, special
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.dpi': 80, 'font.size': 11})
print("Libraries loaded.")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SETUP — Run this cell once. No modification needed.
# Loads the pre-generated GWAS dataset.
# ─────────────────────────────────────────────────────────────────────────────
import os, numpy as np, pandas as pd
from scipy import stats, special
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import warnings; warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 80, 'font.size': 11})

# Locate the data directory, downloading from GitHub if needed (e.g. on Colab).
import os, urllib.request
_NEED = ['gwas_data.npz', 'fly_data.csv']
_LFS  = {'gwas_data.npz', 'sumstats_real.npz'}   # tracked with Git LFS (media URL)
def _has_all(d):
    return d and all(os.path.exists(os.path.join(d, f)) for f in _NEED)
DATA_DIR = next((d for d in ('../data', 'data') if _has_all(d)), None)
if DATA_DIR is None:
    DATA_DIR = 'data'; os.makedirs(DATA_DIR, exist_ok=True)
    for _f in _NEED:
        _dest = os.path.join(DATA_DIR, _f)
        if os.path.exists(_dest):
            continue
        _base = ('https://media.githubusercontent.com/media' if _f in _LFS
                 else 'https://raw.githubusercontent.com')
        _url = f'{_base}/nikbaya/smartbiomed_practicals_2026/main/data/{_f}'
        print(f'Downloading {_f} from GitHub ...')
        urllib.request.urlretrieve(_url, _dest)

if not os.path.exists(os.path.join(DATA_DIR, 'gwas_data.npz')):
    raise FileNotFoundError(
        "Could not find or download gwas_data.npz. Check your internet connection "
        "(the notebook downloads data from GitHub on first run), or ask your instructor."
    )

print("Loading pre-generated GWAS data...")
d = np.load(os.path.join(DATA_DIR, 'gwas_data.npz'), allow_pickle=True)
_G    = d['G_raw']                        # int8: 0/1/2, -1=missing (compact storage)
G_raw = np.where(_G == -1, np.nan, _G.astype(np.float32))  # float32 NaN for missing
del _G
pos        = d['pos']                    # variant positions (kbp)
rsids      = d['rsids']                  # variant RSIDs
age        = d['age']
sex        = d['sex']                    # 0=female, 1=male
y_cont     = d['y_cont']                 # continuous phenotype (standardised)
y_poly     = d['y_poly']                 # polygenic phenotype (h²≈0.02, uncorrelated with y_cont)
y_bin      = d['y_bin']                  # binary phenotype (0/1, ~10% cases)
true_betas = d['true_betas']             # true causal effect sizes
dom_idx_qc = int(d['dom_idx_qc'][0])     # post-QC column index of dominant locus
rec_idx_qc = int(d['rec_idx_qc'][0])     # post-QC column index of recessive locus
fly_df     = pd.read_csv(os.path.join(DATA_DIR, 'fly_data.csv'))
N, M_raw = G_raw.shape
CHROM = 1
print(f"Loaded: {N:,} samples × {M_raw:,} variants (chr{CHROM}, pre-QC)")
print(f"  Continuous trait: standardised liability (h²≈0.035)")
print(f"  Binary trait: {y_bin.sum():,} cases ({100*y_bin.mean():.1f}%, liability threshold)")

N, M_raw = G_raw.shape
print(f"\nReady: N={N:,} samples, M_raw={M_raw:,} variants (pre-QC)")


In [ ]:
# ── Phenotype and covariate distributions ────────────────────────────────────
# Three traits to characterise:
#   y_cont — continuous (h²≈0.035, few large-effect causal variants)
#   y_poly — continuous (h²≈0.02, fully polygenic — every variant has a tiny effect)
#   y_bin  — binary (~10% cases, derived from y_cont via liability threshold)
fig, axes = plt.subplots(2, 3, figsize=(14, 7))

for ax, y, col, lbl in zip(
        [axes[0,0], axes[0,1]],
        [y_cont, y_poly],
        ['steelblue', '#59a14f'],
        ['Continuous (h²≈0.035)', 'Polygenic (h²≈0.02)']):
    ax.hist(y, bins=60, color=col, edgecolor='white', linewidth=0.3)
    ax.set_xlabel('Standardised phenotype'); ax.set_ylabel('Individuals')
    ax.set_title(lbl)

axes[0,2].bar(['Control (0)', 'Case (1)'],
              [(y_bin==0).sum(), (y_bin==1).sum()],
              color=['#4e79a7', '#e15759'], width=0.5)
axes[0,2].set_ylabel('Count')
axes[0,2].set_title(f'Binary trait ({100*y_bin.mean():.0f}% cases)')

axes[1,0].scatter(y_cont[::5], y_poly[::5], s=2, alpha=0.3, color='grey')
axes[1,0].set_xlabel('y_cont'); axes[1,0].set_ylabel('y_poly')
rho_cp = np.corrcoef(y_cont, y_poly)[0,1]
axes[1,0].set_title(f'y_cont vs y_poly  (r={rho_cp:.2f})')

axes[1,1].hist(age, bins=40, color='#f28e2b', edgecolor='white', linewidth=0.3)
axes[1,1].set_xlabel('Age'); axes[1,1].set_title('Age distribution')

axes[1,2].boxplot([y_cont[y_bin==0], y_cont[y_bin==1]], labels=['Control', 'Case'],
                  patch_artist=True,
                  boxprops=dict(facecolor='#4e79a7'), medianprops=dict(color='white'))
axes[1,2].set_ylabel('y_cont'); axes[1,2].set_title('y_cont by case/control status')

plt.suptitle('Dataset overview — three phenotypes', fontsize=13, y=1.01)
plt.tight_layout(); plt.show()
print(f"N = {N:,} individuals  |  M_raw = {M_raw:,} variants (pre-QC)  |  "
      f"Cases: {y_bin.sum():,} ({100*y_bin.mean():.0f}%)")
print(f"Pearson r(y_cont, y_poly) = {rho_cp:.3f}  "
      f"— in Session 2 you will test which trait pairs are genetically correlated.")


## Part 1: GWAS Quality Control

Before running a GWAS, we need to filter out low-quality variants.
Standard filters remove variants with:
- **High missingness** (call rate < 95%, i.e., >5% samples missing a genotype)
- **Low MAF** (minor allele frequency < 1%; rare variants have low power)
- **HWE violation** (excess homozygosity often indicates genotyping error)

In the following exercises, `G_raw` is the raw genotype matrix (N × M_raw).
Genotypes are coded as 0/1/2 (copies of the effect allele); `NaN` = missing.


In [ ]:
# ── Exercise 1.1: Per-variant missingness ─────────────────────────────────────
# Missingness rate = fraction of samples with a missing genotype (NaN).
# A variant "fails" QC if its missingness rate exceeds 5%.

# YOUR CODE HERE
# Hint: np.isnan(G_raw).mean(axis=0) gives the fraction of NaN per column
miss_rate = ???                         # shape (M_raw,)

# ─────────────────────────────────────────────────────────────────────────────
print(f"Variants with missingness > 5%: {(miss_rate > 0.05).sum():,}")

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(miss_rate, bins=50, color='steelblue', edgecolor='white', linewidth=0.4)
ax.axvline(0.05, color='red', linestyle='--', label='5% threshold')
ax.set_xlabel('Missingness rate'); ax.set_ylabel('Variants (log scale)')
ax.set_yscale('log'); ax.set_title('Per-variant missingness distribution'); ax.legend()
plt.tight_layout(); plt.show()


In [ ]:
# ── Exercise 1.2: Minor Allele Frequency ─────────────────────────────────────
# MAF = frequency of the less common allele.
# Genotype dosage 0/1/2 → ALT allele frequency = mean(G) / 2.

# YOUR CODE HERE
# Hint: use np.nanmean to ignore NaN; compute frequency of ALT allele, then take the minor
alt_freq = ???                          # ALT allele frequency, shape (M_raw,)
maf      = ???                          # MAF = min(alt_freq, 1 - alt_freq)

# ─────────────────────────────────────────────────────────────────────────────
print(f"Variants with MAF < 1%:  {(maf < 0.01).sum():,}")
print(f"Variants with MAF < 5%:  {(maf < 0.05).sum():,}")

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(maf, bins=60, color='teal', edgecolor='white', linewidth=0.4)
ax.axvline(0.01, color='red',    linestyle='--', label='1% threshold')
ax.axvline(0.05, color='orange', linestyle='--', label='5% threshold')
ax.set_xlabel('MAF'); ax.set_ylabel('Variants')
ax.set_title('MAF distribution (pre-QC)'); ax.legend()
plt.tight_layout(); plt.show()


### Exercise 1.3: Hardy-Weinberg Equilibrium (HWE) test

Under HWE, genotype counts follow: $n_{AA} \approx n p^2$, $n_{AB} \approx n 2pq$, $n_{BB} \approx n q^2$,
where $p$ = ALT allele frequency and $q = 1-p$.

Violations can indicate genotyping errors (excess homozygosity is most common).
Standard GWAS threshold: HWE $p < 10^{-6}$ (remove variants that fail).

We'll implement a chi-squared test and compare it to a more accurate mid-p exact test (provided).


In [ ]:
# ── Exercise 1.3: HWE chi-squared test (vectorised) ──────────────────────────
def compute_hwe_chisq(G):
    """
    Vectorised chi-squared test for HWE across all variants.
    Operates column-wise with numpy — no Python loop needed.
    Returns p-values of shape (M,).
    """
    G_int  = np.where(np.isnan(G), -1, G).astype(int)
    n_samp = (G_int >= 0).sum(0).astype(float)   # non-missing count per variant
    n_AA   = (G_int == 0).sum(0).astype(float)
    n_AB   = (G_int == 1).sum(0).astype(float)
    n_BB   = (G_int == 2).sum(0).astype(float)

    # YOUR CODE — all vectorised over all M variants simultaneously
    # ALT allele frequency: each copy of the ALT allele (BB = 2, AB = 1) contributes
    p      = ???   # (2*n_BB + n_AB) / (2*n_samp)

    # Expected genotype counts under HWE: P(0 ALT) = (1-p)², P(1 ALT) = 2p(1-p), P(2 ALT) = p²
    exp_AA = ???
    exp_AB = ???
    exp_BB = ???

    # Chi-squared: sum (obs - exp)² / exp, 1 degree of freedom
    chi2   = ???

    return stats.chi2.sf(chi2, df=1)

hwe_chisq = compute_hwe_chisq(G_raw)

# ─────────────────────────────────────────────────────────────────────────────
# Provided: alternative HWE test based on heterozygote deviation
def compute_hwe_midp(G):
    """
    Vectorised heterozygosity-based chi-squared test for HWE.
    Uses the heterozygote deviation formula: chi2 = n*(obs_het - exp_het)^2 / exp_het.
    Returns p-values for each variant (M,).
    """
    G_int  = np.where(np.isnan(G), -1, G).astype(int)
    n_samp = (G_int >= 0).sum(0).astype(float)
    n_AA   = (G_int == 0).sum(0).astype(float)
    n_AB   = (G_int == 1).sum(0).astype(float)
    # p_hat = REF allele freq; exp_het = 2*p*(1-p) is symmetric
    p_hat   = (2*n_AA + n_AB) / (2*n_samp + 1e-15)
    obs_het = n_AB / (n_samp + 1e-15)
    exp_het = 2 * p_hat * (1 - p_hat)
    chi2    = n_samp * (obs_het - exp_het)**2 / (exp_het + 1e-15)
    return stats.chi2.sf(chi2, df=1)

hwe_midp = compute_hwe_midp(G_raw)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, hwe_p, title in zip(axes, [hwe_chisq, hwe_midp],
                              ['3-class chi-squared', 'Heterozygosity test']):
    ax.scatter(-np.log10(hwe_midp + 1e-15), -np.log10(hwe_p + 1e-15),
               alpha=0.2, s=1, color='steelblue', rasterized=True)
    lim = max((-np.log10(hwe_midp + 1e-15)).max(), (-np.log10(hwe_p + 1e-15)).max()) * 1.05
    ax.plot([0, lim], [0, lim], 'r--', linewidth=1)
    ax.set_title(title)
axes[0].set_xlabel('-log10(het test)'); axes[0].set_ylabel('-log10(3-class chi-sq)')
axes[1].set_xlabel('-log10(het test)'); axes[1].set_ylabel('-log10(het test)')
plt.suptitle('HWE test comparison'); plt.tight_layout(); plt.show()

print(f"Variants failing HWE (chi-sq, p<1e-6): {(hwe_chisq < 1e-6).sum():,}")
print(f"Variants failing HWE (het test, p<1e-6): {(hwe_midp < 1e-6).sum():,}")
print("Q: Where do the two tests agree? Where do they differ?")


In [ ]:
# ── Apply QC filters ──────────────────────────────────────────────────────────
# Standard GWAS filters (thresholds given; no coding required here)
MISS_THRESH = 0.05     # remove variants with >5% missing
MAF_THRESH  = 0.01     # remove variants with MAF < 1%
HWE_THRESH  = 1e-6     # remove variants with HWE p < 1e-6

pass_miss = miss_rate < MISS_THRESH
pass_maf  = maf       > MAF_THRESH
pass_hwe  = hwe_chisq > HWE_THRESH   # use your chi-sq result

qc_pass = pass_miss & pass_maf & pass_hwe

G_qc  = np.where(np.isnan(G_raw[:, qc_pass]), 0, G_raw[:, qc_pass])  # impute remaining NaN
pos_qc   = pos[qc_pass]
rsids_qc = rsids[qc_pass]
M_qc  = qc_pass.sum()

print("QC Summary")
print(f"  Pre-QC:               {M_raw:>7,} variants")
print(f"  After missingness QC: {pass_miss.sum():>7,} variants")
print(f"  After MAF QC:         {pass_maf.sum():>7,} variants")
print(f"  After HWE QC:         {pass_hwe.sum():>7,} variants")
print(f"  After all filters:    {M_qc:>7,} variants  ← G_qc")

del G_raw  # free ~4 GB; G_qc is the working matrix from here on


## Part 2: Running GWAS — Continuous Trait

We'll use a vectorised OLS function (provided) that runs all M regressions efficiently.
The model for each variant $j$ is:

$$y_i = \mu + \beta_j \cdot G_{ij} + \gamma_1 \cdot \text{age}_i + \gamma_2 \cdot \text{sex}_i + \varepsilon_i$$

where $\hat{\beta}_j$ is the per-allele effect size on the standardised phenotype (in SD units).


In [ ]:
def run_gwas(y, G, covars=None, chunk=5_000):
    """
    Vectorised OLS GWAS: regress phenotype y on each column of G.
    Processes variants in batches to keep peak memory usage low.

    Parameters
    ----------
    y      : (N,)    phenotype (will be mean-centred internally)
    G      : (N, M)  genotype matrix (0/1/2), NaN = missing (treated as mean)
    covars : (N, k)  covariate matrix (optional); age and sex recommended
    chunk  : int     variants per batch (default 5,000)

    Returns
    -------
    betas  : (M,)  per-variant OLS effect size estimate
    ses    : (M,)  standard error of beta
    pvals  : (M,)  two-sided p-value
    """
    N, M = G.shape
    if covars is None:
        C = np.ones((N, 1))
    else:
        C = np.column_stack([np.ones(N), covars])

    # Residualise y on covariates once (cheap)
    Q, _  = np.linalg.qr(C, mode='reduced')
    y_r   = y - Q @ (Q.T @ y)
    ss_y  = float(np.dot(y_r, y_r))
    n_df  = N - C.shape[1] - 1

    betas = np.empty(M); ses = np.empty(M)

    for s in range(0, M, chunk):
        e    = min(s + chunk, M)
        G_c  = G[:, s:e].astype(float)
        mu   = np.nanmean(G_c, axis=0)
        ri, ci = np.where(np.isnan(G_c)); G_c[ri, ci] = mu[ci]   # mean-impute
        G_r  = G_c - Q @ (Q.T @ G_c)                              # residualise
        ss_G = (G_r**2).sum(0)
        b    = G_r.T @ y_r / ss_G
        betas[s:e] = b
        rss  = ss_y - b**2 * ss_G
        ses[s:e]   = np.sqrt(np.clip(rss, 0, None) / n_df / ss_G)

    t_stats = betas / (ses + 1e-300)
    pvals   = 2 * stats.t.sf(np.abs(t_stats), df=n_df)
    return betas, ses, pvals

print('run_gwas() ready.')

In [ ]:
# ── Exercise 2.1: GWAS without covariates ────────────────────────────────────
# Run GWAS for the continuous trait without adjusting for age or sex.

# YOUR CODE HERE — call run_gwas with appropriate arguments
betas_nocov, ses_nocov, pvals_nocov = run_gwas(???, ???)

# ─────────────────────────────────────────────────────────────────────────────
n_sig = (pvals_nocov < 5e-8).sum()
print(f"Genome-wide significant hits (p < 5e-8): {n_sig:,}")
print(f"Min p-value: {pvals_nocov.min():.2e}")


In [ ]:
# ── Exercise 2.2: GWAS with covariates ───────────────────────────────────────
# Now include age and sex as covariates.
# The covariate matrix should have shape (N, 2).

# YOUR CODE HERE
covars = np.column_stack([???])         # age and sex (standardise age!)
betas_cov, ses_cov, pvals_cov = run_gwas(???, ???, ???)

# ─────────────────────────────────────────────────────────────────────────────
# Compare lambda GC before and after covariate adjustment
def lambda_gc(pvals):
    chi2_obs = stats.chi2.isf(np.clip(pvals, 1e-300, 1), df=1)
    return np.median(chi2_obs) / stats.chi2.median(df=1)

print(f"Lambda GC without covariates: {lambda_gc(pvals_nocov):.3f}")
print(f"Lambda GC with covariates:    {lambda_gc(pvals_cov):.3f}")
print("Q: Did adding covariates change lambda GC? What does lambda GC > 1.0 imply?")


In [ ]:
# ── Exercise 2.3: Interpreting beta — strip plot by genotype ──────────────────
# The OLS beta estimates the per-allele shift in phenotype.
# For an additive variant: E[y | g] ≈ μ + g × beta.
# A strip plot makes the genotype-phenotype trend visible on individual level.
import seaborn as sns

j_top    = np.argmin(pvals_cov)
beta_top = betas_cov[j_top]
geno     = G_qc[:, j_top].astype(int)
print(f"Lead variant: {rsids_qc[j_top]}  pos={pos_qc[j_top]:,} kbp  "
      f"beta={beta_top:.4f}  p={pvals_cov[j_top]:.2e}")

# YOUR CODE HERE
means  = ???   # mean phenotype for each of the 3 genotype classes (0, 1, 2)
counts = ???   # number of individuals per class

# ─────────────────────────────────────────────────────────────────────────────
_df = pd.DataFrame({'Genotype': geno, 'Phenotype': y_cont})
_df['Genotype'] = _df['Genotype'].map({0:'AA (0)', 1:'AB (1)', 2:'BB (2)'})

fig, ax = plt.subplots(figsize=(6, 4))
sns.stripplot(data=_df.sample(min(2000, len(_df)), random_state=0),
              x='Genotype', y='Phenotype', alpha=0.25, size=2, jitter=True,
              palette=['#4e79a7','#f28e2b','#e15759'], ax=ax)
ax.plot([0,1,2], means, color='black', linewidth=1.5, zorder=4)
ax.errorbar([0,1,2], means, fmt='_', color='black', markersize=20, linewidth=2,
            capsize=0, zorder=5)
ax.set_ylabel('Phenotype (standardised SD)')
ax.set_title(f'Lead SNP — per-allele shift ≈ {beta_top:.3f} SD')
plt.tight_layout(); plt.show()

print(f"Expected BB−AA ≈ 2 × beta = {2*beta_top:.4f} SD")
print(f"Observed BB−AA:              {means[2]-means[0]:.4f} SD")
print("Q: Is the heterozygote (AB) midway between the two homozygotes? What does that imply?")


## Part 3: GWAS — Binary Trait (Liability Threshold Model)

For binary (case/control) phenotypes, logistic regression is standard.
The model gives a **log-odds ratio** (log-OR) per allele.

The binary phenotype `y_bin` (1=case, 0=control) was derived from the continuous phenotype
via a **liability threshold**: individuals above the 90th percentile of liability are cases (~10% prevalence).
This is the standard liability threshold model for complex diseases.

We provide `run_logistic_gwas_fast()` — a score-test approximation that runs quickly.


In [ ]:
def run_logistic_gwas_fast(y_bin, G, covars=None, chunk=5_000):
    """
    Fast logistic GWAS: score test (no iterative fitting per variant).
    Fits null model once; processes variants in batches to limit memory.

    Returns
    -------
    log_ors : (M,) approximate log-OR
    pvals   : (M,) score test p-values
    """
    from scipy.special import expit
    from scipy.optimize import minimize

    N, M = G.shape
    if covars is None:
        C0 = np.ones((N, 1))
    else:
        C0 = np.column_stack([np.ones(N), covars])
    k0 = C0.shape[1]

    def neg_ll(coef, X, y):
        mu = expit(X @ coef)
        return -np.sum(y * np.log(mu + 1e-15) + (1-y) * np.log(1 - mu + 1e-15))

    res0  = minimize(neg_ll, np.zeros(k0), args=(C0, y_bin), method='L-BFGS-B')
    mu0   = expit(C0 @ res0.x)
    resid = y_bin.astype(float) - mu0
    W0    = mu0 * (1 - mu0)

    # Precompute weighted projection: P_W = C0 @ (C0'WC0)^{-1} C0'W
    WC       = W0[:, None] * C0                              # (N, k0)
    CWWC_inv = np.linalg.inv(C0.T @ WC + 1e-12*np.eye(k0))  # (k0, k0)

    log_ors = np.empty(M); pvals = np.ones(M)

    for s in range(0, M, chunk):
        e    = min(s + chunk, M)
        G_c  = G[:, s:e].astype(float)
        mu   = np.nanmean(G_c, 0); ri, ci = np.where(np.isnan(G_c)); G_c[ri, ci] = mu[ci]
        G_rc = G_c - C0 @ (CWWC_inv @ (WC.T @ G_c))        # weighted residualise
        score  = G_rc.T @ resid
        var_sc = (G_rc**2 * W0[:, None]).sum(0)
        z      = score / np.sqrt(var_sc + 1e-15)
        pvals[s:e]   = 2 * stats.norm.sf(np.abs(z))
        log_ors[s:e] = score / (var_sc + 1e-15)

    return log_ors, pvals

print('run_logistic_gwas_fast() ready.')

In [ ]:
# ── Exercise 3.1: Logistic GWAS for CAD ──────────────────────────────────────
# Run the logistic GWAS for CAD using the fast score test.

# YOUR CODE HERE
covars = np.column_stack([???])    # same covariates as before
log_ors, pvals_bin = run_logistic_gwas_fast(???, ???, ???)

# ─────────────────────────────────────────────────────────────────────────────
n_sig_bin = (pvals_bin < 5e-8).sum()
print(f"Genome-wide significant binary-trait hits: {n_sig_bin:,}")

# Compare LDL and CAD p-values at the same variants
top_ldl_idx = np.argsort(pvals_cov)[:20]
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(-np.log10(pvals_cov[top_ldl_idx]+1e-15),
           -np.log10(pvals_bin[top_ldl_idx]+1e-15),
           alpha=0.8, s=40, color='steelblue')
ax.set_xlabel('-log10(p) continuous trait')
ax.set_ylabel('-log10(p) binary trait')
ax.set_title('Top 20 continuous-trait hits: continuous vs binary p-values')
for i, j in enumerate(top_ldl_idx[:5]):
    ax.annotate(f"rs{j}", ((-np.log10(pvals_cov[j]+1e-15)),
                            (-np.log10(pvals_bin[j]+1e-15))), fontsize=7)
plt.tight_layout(); plt.show()
print("Q: Do the same variants drive both continuous and binary trait associations?")


---

## Challenge Questions

These questions are for fast finishers and are not required in the 45-minute session.
They connect directly to ideas from the lecture.


### Challenge 1: Additive, Dominant, and Recessive Models

The standard GWAS uses an **additive** model: genotype is encoded 0/1/2 (copies of ALT allele),
so the heterozygote AB is midway between AA and BB.

But what if the true effect is **dominant** (AB ≈ BB, one copy is enough) or
**recessive** (AB ≈ AA, two copies are needed)?

Recoding:
- **Dominant**: 0 → 0, 1 → 1, 2 → 1  (any ALT copy counts)
- **Recessive**: 0 → 0, 1 → 0, 2 → 1  (only homozygous ALT counts)

Two variants in this dataset have non-additive effects — one dominant, one recessive.
They may not be the genome-wide lead hit under the additive model!


In [ ]:
# Challenge 1: Find the non-additive loci
import seaborn as sns

# YOUR CODE HERE
# Recode G_qc under dominant and recessive models
G_dom = ???   # 0→0, 1→1, 2→1
G_rec = ???   # 0→0, 1→0, 2→1

covars = np.column_stack([(age - age.mean()) / age.std(), sex])
betas_dom, _, pvals_dom = run_gwas(y_cont, G_dom, covars)
betas_rec, _, pvals_rec = run_gwas(y_cont, G_rec, covars)

# Find the best hit under each non-additive model
j_dom = np.argmin(pvals_dom)
j_rec = np.argmin(pvals_rec)
print(f"Top dominant hit: {rsids_qc[j_dom]}  pos={pos_qc[j_dom]:,} kbp  p={pvals_dom[j_dom]:.2e}")
print(f"Top recessive hit: {rsids_qc[j_rec]}  pos={pos_qc[j_rec]:,} kbp  p={pvals_rec[j_rec]:.2e}")

# Strip plots for each — does the AB group look more like AA or BB?
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, j, title in zip(axes, [j_dom, j_rec], ['Putative dominant', 'Putative recessive']):
    geno = G_qc[:, j].astype(int)
    _df  = pd.DataFrame({'Genotype': geno, 'Phenotype': y_cont})
    _df['Genotype'] = _df['Genotype'].map({0:'AA', 1:'AB', 2:'BB'})
    sns.stripplot(data=_df.sample(min(2000, len(_df)), random_state=0),
                  x='Genotype', y='Phenotype', alpha=0.25, size=2, jitter=True,
                  palette=['#4e79a7','#f28e2b','#e15759'], ax=ax)
    means = [y_cont[geno==g].mean() for g in range(3)]
    ax.plot([0,1,2], means, color='black', linewidth=1.5, zorder=4)
    ax.errorbar([0,1,2], means, fmt='_', color='black', markersize=20, linewidth=2, zorder=5)
    ax.set_title(title); ax.set_ylabel('Phenotype')
plt.tight_layout(); plt.show()
print("Q: For the dominant hit, is AB closer to AA or BB? What about the recessive hit?")


### Challenge 2: Manual LocusZoom plot

A **LocusZoom plot** shows $-\log_{10}(p)$ vs. position for a region around a hit, with points
**coloured by LD** ($r^2$) with the lead variant. It reveals the LD structure that produces the
"tower" you'll see in a Manhattan plot (Session 2) — neighbouring variants are associated only
because they are correlated with the causal one.

We use the covariate-adjusted continuous-trait GWAS (`pvals_cov`) and the genotypes `G_qc`.


In [ ]:
# Challenge 2: Manual LocusZoom plot for the lead locus (continuous trait)

# Region: ±1 Mb around the lead variant
j_lead    = np.argmin(pvals_cov)
pos_lead  = pos_qc[j_lead]
region_mask = np.abs(pos_qc - pos_lead) < 1000   # ±1000 kbp = ±1 Mb

print(f"Lead variant: {rsids_qc[j_lead]}  pos={pos_lead:,} kbp  p={pvals_cov[j_lead]:.2e}")
print(f"Variants in region: {region_mask.sum():,}")

# YOUR CODE HERE
# Compute r² between each variant in the region and the lead variant.
# Hint: r = np.corrcoef(g_lead, G_qc[:, k])[0,1]; r² = r**2
g_lead = G_qc[:, j_lead]
r2 = ???    # shape (region_mask.sum(),)

cmap = plt.cm.RdYlBu_r   # red = high LD, blue = low LD
norm = mcolors.Normalize(vmin=0, vmax=1)
fig, ax = plt.subplots(figsize=(12, 4))
sc = ax.scatter(pos_qc[region_mask] / 1000,
                -np.log10(pvals_cov[region_mask] + 1e-300),
                c=r2, cmap=cmap, norm=norm, s=20, zorder=3)
ax.scatter([pos_lead/1000], [-np.log10(pvals_cov[j_lead]+1e-300)],
           s=100, marker='D', color='black', zorder=5, label='Lead SNP')
plt.colorbar(sc, ax=ax, label=r'$r^2$ with lead SNP')
ax.axhline(-np.log10(5e-8), color='red', linestyle='--', linewidth=1, label='5×10⁻⁸')
ax.set_xlabel('Position on chr1 (Mb)'); ax.set_ylabel(r'$-\log_{10}(p)$')
ax.set_title('LocusZoom: lead locus on chr1 (±1 Mb)')
ax.legend(fontsize=8); plt.tight_layout(); plt.show()
print("Q: Do the high-r² (red) variants form the tower? What happens to p as r² drops?")


### Challenge 3: Drosophila Linkage Analysis (Hard!)

*(Inspired by Sturtevant, 1913 — the first genetic map.)*

We have 2,000 fly offspring from a test cross:
- **Mother**: heterozygous for 6 X-linked traits (carrier) + 2 autosomal traits
- **Father**: hemizygous wild-type

The dataset `fly_df` has columns: `sex` (1=male, 0=female) and 8 trait columns.
You are told there are 6 X-linked traits and 2 autosomal traits — but the column names
don't tell you which is which.

**Your tasks**:
1. Identify which traits are X-linked (hint: look at trait frequency in males vs females).
2. For the X-linked traits, compute pairwise recombination frequencies.
3. Use the recombination frequencies to infer the order and spacing of the 6 genes on the chromosome.
4. *(Extension)* Use `simulate_cross()` to design a new cross and verify your inferred map.


In [ ]:
# Challenge 2, Part A: Identify X-linked traits
# For X-linked recessive traits in a test cross: males show the trait ~50% of the time,
# females show it ~0% of the time (they are carriers).
# For autosomal traits: frequency is similar in males and females.

fly_df.head()


In [ ]:
# Challenge 2, Part A (continued)
trait_cols = [c for c in fly_df.columns if c.startswith('trait_')]

# YOUR CODE HERE
# For each trait, compute the mean (= frequency of showing the trait) in males and females separately
# Hint: fly_df.groupby('sex')[trait_cols].mean()
freq_by_sex = ???

print(freq_by_sex.T)
print("\nWhich traits are X-linked? (large difference between males and females)")
x_linked_traits = ???   # list of column names that are X-linked


In [ ]:
# Challenge 2, Part B: Pairwise recombination frequencies
# For X-linked traits, recombination frequency between genes A and B =
# fraction of MALE offspring where the phenotype for A DIFFERS from phenotype for B.
# (Non-recombinants have all traits in coupling; recombinants show a 'break' in the pattern.)

males = fly_df[fly_df['sex'] == 1]

# YOUR CODE HERE
# Compute the pairwise recombination frequency matrix (n_X × n_X)
# Hint: for traits i and j, recomb_freq[i,j] = fraction of males where trait_i != trait_j
n_x = len(x_linked_traits)
recomb_freq = np.zeros((n_x, n_x))
for i, ti in enumerate(x_linked_traits):
    for j, tj in enumerate(x_linked_traits):
        recomb_freq[i, j] = ???

print("Pairwise recombination frequencies (males only):")
pd.DataFrame(recomb_freq, index=x_linked_traits, columns=x_linked_traits).round(3)


In [ ]:
# Challenge 2, Part C: Infer gene order
# The pair with the SMALLEST recombination frequency are the CLOSEST together.
# Iteratively build up the genetic map by placing genes relative to each other.
#
# Hint: Start with the two most closely linked genes, then find which other gene
# is closest to one end or the other.

# YOUR CODE HERE
# 1. Find the gene pair with the smallest recombination frequency
# 2. Build up the map order step by step
# 3. Convert recombination frequencies to genetic distances (Haldane map function)
#    d = -50 * log(1 - 2r) cM    [Haldane]

def haldane_d(r):
    """Convert recombination frequency r to map distance in cM."""
    return ???

# Print your inferred gene order and map distances


In [ ]:
# Challenge 2, Part D: Verify with simulate_cross()
# Use the provided simulate_cross() to test your inferred gene order.
# Design a cross using your inferred map and check if the recombination frequencies match.
def simulate_cross(female_X_genotype, male_X_genotype, n_offspring=500,
                   gene_positions_cM=None, autosomal_traits=None, seed=None):
    """
    Simulate offspring from a Drosophila cross.

    Parameters
    ----------
    female_X_genotype : array of shape (2, n_X_genes)
        Two X chromosomes of the female parent.
        Row 0 = first X haplotype, Row 1 = second X haplotype.
        Values 0 = wild-type, 1 = mutant allele.
    male_X_genotype   : array of shape (1, n_X_genes)
        Single X chromosome of the male parent.
    n_offspring       : int
        Number of offspring to simulate.
    gene_positions_cM : array of length n_X_genes
        Genetic positions in centiMorgans (default: equally spaced at 10 cM).
    autosomal_traits  : dict {trait_name: freq}
        Optional autosomal traits with given allele frequencies.
    seed              : int, optional

    Returns
    -------
    pd.DataFrame with columns: sex, trait_X1, trait_X2, ..., [autosomal traits]
    """
    if seed is not None:
        np.random.seed(seed)
    n_X = female_X_genotype.shape[1]
    if gene_positions_cM is None:
        gene_positions_cM = np.arange(n_X) * 10.0
    pos_M = np.array(gene_positions_cM) / 100   # cM → Morgans

    # Simulate n_offspring
    fly_sex = np.random.randint(0, 2, n_offspring)   # 0=female, 1=male
    X_pheno = np.zeros((n_offspring, n_X), dtype=np.int8)

    for i in range(n_offspring):
        # Female transmits one X (with recombination)
        start_hap = np.random.randint(0, 2)   # which of the 2 female X chromosomes
        curr = start_hap
        transmitted = np.zeros(n_X, dtype=np.int8)
        transmitted[0] = female_X_genotype[curr, 0]
        for j in range(1, n_X):
            dist = pos_M[j] - pos_M[j-1]
            n_xo = np.random.poisson(dist)
            curr = (curr + n_xo) % 2
            transmitted[j] = female_X_genotype[curr, j]

        if fly_sex[i] == 1:  # male: X from mum, Y from dad → shows mutant if allele = 1
            X_pheno[i] = transmitted
        else:                # female: X from mum + X from dad → heterozygous, doesn't show recessive
            X_pheno[i] = 0

    result = pd.DataFrame(X_pheno, columns=[f'trait_X{g+1}' for g in range(n_X)])
    result.insert(0, 'sex', fly_sex)

    if autosomal_traits:
        for name, freq in autosomal_traits.items():
            result[name] = np.random.binomial(1, freq, n_offspring)

    return result

# Example: heterozygous female for all 6 traits (mutant/WT) × WT male
female_X = np.array([[1, 1, 1, 1, 1, 1],   # mutant chromosome
                      [0, 0, 0, 0, 0, 0]])  # WT chromosome

# YOUR CODE HERE: set gene positions based on your inferred map
inferred_positions_cM = ???    # list of 6 positions in cM

offspring = simulate_cross(female_X, np.array([[0, 0, 0, 0, 0, 0]]),
                           n_offspring=2000,
                           gene_positions_cM=inferred_positions_cM,
                           seed=99)
print("Simulated recombination frequencies (males only):")
sim_males = offspring[offspring['sex'] == 1]
x_cols = [c for c in sim_males.columns if c.startswith('trait_X')]
sim_r = pd.DataFrame([[  (sim_males[ti] != sim_males[tj]).mean()
                          for tj in x_cols] for ti in x_cols],
                     index=x_cols, columns=x_cols)
print(sim_r.round(3))
